# AOP-ProCon — Step 5 **(build v4)**: Evaluation

Evaluates the Step-4 v3 checkpoints on **held-out fold 1** (queries) against **fold 0** (gallery), and runs the supervised mechanism probe. This is a **rewrite**, not a patch of the old notebook, because three facts about the new data make the old protocol unreliable:

| What the data now shows | Consequence for the old Step 5 |
|---|---|
| In the **strict** pool all 18 MC-only peptides are in fold 1 and **none** in fold 0 (fold 0 has 55 dual and 283 FRS-only peptides). | The old metric ("relevant = shares ≥ 1 label", MC-only queries) had **≈ 18 informative queries** and a gallery whose relevant items are all dual peptides. |
| Step 4 shows the saved checkpoints are close to **embedding collapse** (mean pairwise cosine 0.92–1.00 at 12 of 14 checkpoints), and the loss is the same for real and random pairs. | A retrieval number from a trained model is meaningless without baselines that are *not* trained: raw ESM-2, amino-acid composition, length. |
| The earlier "ESM-2 ≈ amino-acid-composition baseline" audit (2026-10-16) was never built into the notebook. | The probe now includes composition, descriptor and length baselines, **grouped by `homology_group`**, with a leaky random-fold run next to it to *measure* the leakage. |
| **(v4, STEP 08 findings N-19 / N-21 / N-22 / N-23)** Every FRS-only peptide comes from AnOxPePred, so the single binary feature `source_group` already predicts MC-active as well as ESM-2; a single `GroupKFold` split moves the probe AUROC by up to 0.06; the pooled out-of-fold AUROC is biased; the 1-draw noise row is not chance. | The probe is rebuilt: **repeated grouped CV, fold-mean AUROC, SOURCE and length confound baselines, a 20-seed noise null band** (section 4). The retrieval part (sections 1-3) is unchanged. |

### What is evaluated
* **MC retrieval** (the only retrieval task that can carry information): queries = MC-active peptides of fold 1 (MC-only + dual); the gallery is fold 0 of the same experiment mode; a gallery item is relevant if it is MC-active. The chance level is computed exactly (Recall@K) and by permutation (AP). FRS retrieval is also computed so that its saturation (chance Recall@5 ≈ 1) is shown, not assumed.
* **Representations:** the 14 trained heads **and** untrained baselines — raw ESM-2 650M / 150M, 20-D amino-acid composition, 14-D descriptors, length only, random.
* **Statistics:** per-query metrics, bootstrap 95 % CI over queries, **paired** differences (same queries) with their own CI, Wilcoxon signed-rank p-values with Holm correction over the comparison family. "No difference detected" is never reported as "equivalent".
* **Held-out collapse diagnostics:** `mean_cos` and `eff_rank` of the fold-1 embeddings (Step 4 measured them on the training pool).
* **Supervised probe** (MC-active vs FRS-only), repeated (5 folds × 10) grouped CV on `homology_group`, fold-mean AUROC, four ESM-2 scales, baselines and the SOURCE / LENGTH confound controls, in the three experiment modes; a repeated random-fold run quantifies how much leakage would inflate the result.

### Limits that apply to everything below
One seed per model (42) and one fold direction (train fold 0 → evaluate fold 1). Sample sizes are small (tens of queries); differences smaller than their paired CI are not findings. Strict-mode MC training signal comes from dual peptides only (no MC-only peptide is in fold 0).

## Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

BASE = "/content/drive/MyDrive/AOP-ProCon"

In [ ]:
import os, json, hashlib, itertools, math, datetime
from importlib import metadata
import numpy as np, pandas as pd, torch, torch.nn.functional as F
from scipy.stats import wilcoxon
from scipy.special import comb
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, average_precision_score
from sklearn.model_selection import GroupKFold, StratifiedKFold
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt

PROC = f"{BASE}/data/processed"; EMB = f"{BASE}/data/embeddings"; RES = f"{BASE}/results"
os.makedirs(RES, exist_ok=True)
device = "cpu"; SEED = 42; N_BOOT = 2000
pd.set_option("display.width", 220); pd.set_option("display.max_columns", 30)

def _ver(pkg):
    try: return metadata.version(pkg)
    except Exception: return "n/a"
VERSIONS = {"python": __import__("sys").version.split()[0], "torch": torch.__version__, "numpy": np.__version__,
            "pandas": pd.__version__, "scikit-learn": _ver("scikit-learn"), "scipy": _ver("scipy")}
print(VERSIONS)

def s2_fingerprint(s2_df):
    lines = []
    for r in s2_df.sort_values("sequence").itertuples():
        lines.append("\t".join([r.sequence, r.labels,
                                "" if pd.isna(r.homology_group) else str(int(r.homology_group)),
                                "" if pd.isna(r.lso_fold) else str(int(r.lso_fold)),
                                f"{float(r.sample_weight):.1f}"]))
    return hashlib.md5("\n".join(lines).encode()).hexdigest()[:16]

def sha256_file(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""): h.update(chunk)
    return h.hexdigest()

---
## 0. Load, provenance gates, evaluation sets

Everything must come from the same S2 / Step 2 / Step 3 / Step 4 run: S2 fingerprint, embedding and descriptor SHA-256 (Step 2 manifest), pair-file SHA-256 (Step 3 manifest, stamped into every checkpoint by Step 4). A mismatch stops the notebook.

In [ ]:
EXPECTED_S2_FINGERPRINT = "47d9c2a2fff63eb1"
aop = pd.read_csv(f"{PROC}/AOP-BenchPos_S2.csv")
desc = pd.read_csv(f"{PROC}/descriptors.csv")
need = {"sequence", "labels", "sample_weight", "lso_fold", "cluster_40", "homology_group", "strict_evidence", "homology_resolvable", "source_group"}
assert need <= set(aop.columns) and aop["sequence"].is_unique and desc["sequence"].is_unique
assert set(aop["sequence"]) == set(desc["sequence"])
desc = desc.set_index("sequence").loc[aop["sequence"]].reset_index()
S2_FP = s2_fingerprint(aop)
m2 = json.load(open(f"{PROC}/step2_manifest.json")); pm = json.load(open(f"{PROC}/pairs_manifest.json"))
assert S2_FP == m2["s2_fingerprint"] == pm["s2_fingerprint"], "S2 / Step 2 / Step 3 fingerprints disagree"
assert S2_FP == EXPECTED_S2_FINGERPRINT, f"S2 fingerprint {S2_FP} is not the validated Step 1 build v3"
assert pm.get("build", "").startswith("v4")
assert sha256_file(f"{PROC}/descriptors.csv") == m2["sha256"]["descriptors.csv"]
MODES, PRIMARY_MODE = pm["modes"], pm["primary_mode"]
for key, h in pm["sha256"].items():
    assert sha256_file(f"{PROC}/pairs/{key}") == h, f"pairs/{key} differs from the file Step 3 recorded"
print("S2", S2_FP, "| descriptors + 12 pair files agree with the manifests | modes:", list(MODES), "| primary:", PRIMARY_MODE)

SCALES = {"35M": 480, "150M": 640, "650M": 1280, "3B": 2560}
ESM = {}
for sc, dim in SCALES.items():
    a = np.load(f"{EMB}/esm2_{sc}.npy")
    assert sha256_file(f"{EMB}/esm2_{sc}.npy") == m2["sha256"][f"esm2_{sc}.npy"], f"esm2_{sc}.npy differs from Step 2's record"
    assert a.shape == (len(aop), dim) and np.isfinite(a).all()
    assert open(f"{EMB}/esm2_{sc}_sequences.txt").read().splitlines() == aop["sequence"].tolist()
    ESM[sc] = dict(zip(aop["sequence"], a))
AA = "ACDEFGHIKLMNPQRSTVWY"
AAC = {s: np.array([s.count(a) / len(s) for a in AA]) for s in aop["sequence"]}
DESC = dict(zip(desc["sequence"], desc[[c for c in desc.columns if c != "sequence"]].values))
DCOLS = [c for c in desc.columns if c != "sequence"]
LENGTH = {s: len(s) for s in aop["sequence"]}

mech = aop[aop["labels"].str.contains("FRS|MC")].reset_index(drop=True)
assert mech["lso_fold"].notna().all() and mech["homology_group"].notna().all()
for col in ("sequence", "cluster_40", "homology_group"):
    assert mech.groupby(col)["lso_fold"].nunique().max() == 1, f"{col} spans both folds"
gen_seqs = aop[aop["labels"] == "GEN"]["sequence"].tolist()

def mode_pool(mode):
    d = mech
    if MODES[mode]["strict"]: d = d[d["strict_evidence"].astype(bool)]
    if MODES[mode]["ge4"]:    d = d[d["homology_resolvable"].astype(bool)]
    return d.reset_index(drop=True)
POOL = {m: mode_pool(m) for m in MODES}
EVAL = {}
for m, d in POOL.items():
    q, g = d[d.lso_fold == 1].reset_index(drop=True), d[d.lso_fold == 0].reset_index(drop=True)
    for col in ("sequence", "cluster_40", "homology_group"):
        assert not (set(q[col]) & set(g[col])), f"{m}: query/gallery share {col}"
    EVAL[m] = {"query": q, "gallery": g}
rows = []
for m, e in EVAL.items():
    for part in ("query", "gallery"):
        vc = e[part]["labels"].value_counts()
        rows.append({"mode": m, "set": part, "n": len(e[part]), "FRS": int(vc.get("FRS", 0)), "FRS;MC": int(vc.get("FRS;MC", 0)), "MC": int(vc.get("MC", 0))})
print(pd.DataFrame(rows).to_string(index=False))

### 0b. Load the 14 checkpoints (and the optional 15th)

Each checkpoint carries the S2 fingerprint and the SHA-256 of all 12 pair files it was trained on; both must equal the manifests. The input representation of each model is rebuilt exactly as in Step 4 (HP2-Fusion: descriptor scaler fitted on the primary-mode fold-0 peptides).

In [ ]:
prim_f0 = EVAL[PRIMARY_MODE]["gallery"]
_fit = aop["sequence"].isin(set(prim_f0["sequence"])).values
FUSED_SCALER = StandardScaler().fit(desc[DCOLS].values[_fit])
_ds = FUSED_SCALER.transform(desc[DCOLS].values)
FUSED = {s: np.concatenate([ESM["650M"][s], _ds[i]]) for i, s in enumerate(desc["sequence"])}
LOOKUP = {"650M": ESM["650M"], "150M": ESM["150M"], "fused": FUSED}

MODEL_SPECS = [  # name, file, input, optional
    ("Primary", "model_primary.pt", "650M", False), ("HP1-Random", "model_hp1_random.pt", "650M", False),
    ("HP2-Fusion", "model_hp2_fusion.pt", "fused", False), ("Primary-Balanced", "model_primary_balanced.pt", "650M", False),
    ("ARI-Checkpoint-Only", "model_ari_checkpoint_only.pt", "650M", False), ("k-NN-Pairing", "model_knn_pairing.pt", "650M", False),
    ("k-NN+ARI-Checkpoint", "model_knn_ari_checkpoint.pt", "650M", False), ("Primary-150M", "model_primary_150m.pt", "150M", False),
    ("k-NN-150M", "model_knn_150m.pt", "150M", False), ("ARI-Checkpoint-Floored", "model_ari_checkpoint_floored.pt", "650M", False),
    ("Primary-Extended", "model_primary_extended.pt", "650M", False), ("Primary-StrictGE4", "model_primary_strict_ge4.pt", "650M", False),
    ("Primary-MechEqual", "model_primary_mechequal.pt", "650M", False), ("Primary-MaxSpread", "model_primary_maxspread.pt", "650M", False),
    ("MechEqual-MaxSpread", "model_mechequal_maxspread.pt", "650M", True),
]
MODELS = {}
for name, fn, inp, optional in MODEL_SPECS:
    path = f"{RES}/{fn}"
    if not os.path.exists(path):
        assert optional, f"missing checkpoint {fn} -- run Step 4 first"
        print(f"(optional checkpoint {fn} not found -- skipped)"); continue
    ck = torch.load(path, map_location=device, weights_only=False)
    assert ck["s2_fingerprint"] == S2_FP, f"{name}: trained on another S2"
    assert ck["pair_hashes"] == pm["sha256"], f"{name}: trained on pair files other than the ones Step 3 recorded"
    head = torch.nn.Linear(ck["input_dim"], 128); head.load_state_dict(ck["head_state_dict"]); head.eval()
    MODELS[name] = {"head": head, "ckpt": ck, "input": inp, "mode": ck["pair_mode"]}
    assert ck["input_dim"] == len(next(iter(LOOKUP[inp].values())))
    print(f"{name:<24} mode={ck['pair_mode']:<11} epoch={ck['checkpoint_epoch']:<3} saved_min_dist={ck['min_proto_dist_saved']:.4f} "
          f"mean_cos(train pool)={ck['mean_cos_at_ckpt']:.3f} eff_rank={ck['eff_rank_at_ckpt']:.1f}")

def embed_model(name, seqs):
    m = MODELS[name]
    raw = torch.tensor(np.stack([LOOKUP[m["input"]][s] for s in seqs]), dtype=torch.float32)
    with torch.no_grad(): out = m["head"](raw)
    return F.normalize(out, dim=-1).numpy()

---
## 1. Representations and the retrieval protocol

Every representation is turned into a **similarity matrix** between the fold-1 queries and the fold-0 gallery of an experiment mode:

| representation | similarity |
|---|---|
| trained head (14 models) | cosine of the 128-D projection |
| `RAW 650M`, `RAW 150M` | cosine of the frozen ESM-2 vector (no training) |
| `AAC-20` | cosine of the 20-D amino-acid composition |
| `DESC-14` | cosine of the 14 descriptors, standardised with a scaler fitted on the mode's **gallery** peptides |
| `LENGTH` | minus the absolute length difference |
| `RANDOM` | random scores (permutation chance level) |

**Task.** For a target mechanism T ∈ {MC, FRS}: queries = fold-1 peptides whose label set contains T; a gallery peptide is relevant if its label set contains T. Metrics per query: Average Precision (AP), Recall@5 and Recall@10 (= any relevant item in the top K). Chance Recall@K is exact: `1 − C(N−R, K) / C(N, K)`; chance AP is the mean over 500 random orderings. A task whose chance Recall@5 is ≥ 0.99 **cannot carry information** and is flagged.

In [ ]:
def unit(a): return a / np.linalg.norm(a, axis=1, keepdims=True)

def base_sims(rep, q_df, g_df, rng):
    qs, gs = q_df["sequence"].tolist(), g_df["sequence"].tolist()
    if rep == "RAW 650M": return unit(np.stack([ESM["650M"][s] for s in qs])) @ unit(np.stack([ESM["650M"][s] for s in gs])).T
    if rep == "RAW 150M": return unit(np.stack([ESM["150M"][s] for s in qs])) @ unit(np.stack([ESM["150M"][s] for s in gs])).T
    if rep == "AAC-20":   return unit(np.stack([AAC[s] for s in qs])) @ unit(np.stack([AAC[s] for s in gs])).T
    if rep == "DESC-14":
        sc = StandardScaler().fit(np.stack([DESC[s] for s in gs]))
        return unit(sc.transform(np.stack([DESC[s] for s in qs]))) @ unit(sc.transform(np.stack([DESC[s] for s in gs]))).T
    if rep == "LENGTH":
        lq, lg = np.array([LENGTH[s] for s in qs]), np.array([LENGTH[s] for s in gs])
        return -np.abs(lq[:, None] - lg[None, :]).astype(float) + 1e-6 * rng.rand(len(qs), len(gs))      # tiny jitter: ties broken at random
    raise KeyError(rep)

BASELINES = ["RAW 650M", "RAW 150M", "AAC-20", "DESC-14", "LENGTH"]

def per_query(S, q_df, g_df, target):
    rel_g = g_df["labels"].str.contains(target).values.astype(int)
    qi = np.where(q_df["labels"].str.contains(target).values)[0]
    rows = []
    for i in qi:
        s = S[i]; order = np.argsort(-s, kind="stable"); r = rel_g[order]
        rows.append({"sequence": q_df["sequence"].iloc[i], "AP": average_precision_score(rel_g, s),
                     "R@5": float(r[:5].any()), "R@10": float(r[:10].any())})
    return pd.DataFrame(rows).set_index("sequence")

def chance(g_df, target, n_perm=500, seed=0):
    rel = g_df["labels"].str.contains(target).values.astype(int); N, R = len(rel), int(rel.sum())
    rec = {k: (1.0 if N - R < k else float(1 - comb(N - R, k, exact=True) / comb(N, k, exact=True))) for k in (5, 10)}
    rng = np.random.RandomState(seed)
    ap = float(np.mean([average_precision_score(rel, rng.rand(N)) for _ in range(n_perm)]))
    return {"N_gallery": N, "relevant": R, "chance_R@5": rec[5], "chance_R@10": rec[10], "chance_AP": ap}

def boot_mean_ci(x, n=N_BOOT, seed=0):
    x = np.asarray(x, float); rng = np.random.RandomState(seed)
    b = x[rng.randint(0, len(x), (n, len(x)))].mean(1)
    return float(np.percentile(b, 2.5)), float(np.percentile(b, 97.5))

def spread(E):
    z = unit(E); n = len(z)
    mean_cos = float(((z @ z.T).sum() - n) / (n * (n - 1)))
    s2 = np.linalg.svd(z - z.mean(0, keepdims=True), compute_uv=False) ** 2
    p = s2[s2 > 0] / s2.sum()
    return mean_cos, float(np.exp(-(p * np.log(p)).sum()))

CHANCE = {(m, t): chance(EVAL[m]["gallery"], t) for m in MODES for t in ("MC", "FRS")}
print(pd.DataFrame([{"mode": m, "task": t, **v, "carries_information": v["chance_R@5"] < 0.99} for (m, t), v in CHANCE.items()]).round(4).to_string(index=False))
print("\nThe FRS task is saturated in every mode (nearly every gallery peptide is FRS-active); only MC retrieval can separate representations.")

---
## 2. MC retrieval — every representation, in its own experiment mode

Trained models are evaluated in the mode they were trained in (Models 1–10, 13–15: `strict`; 11: `extended`; 12: `strict_ge4`). The untrained baselines are evaluated in **all three** modes so that every trained model has a baseline on exactly the same queries and gallery.

In [ ]:
rng_len = np.random.RandomState(SEED)
PQ = {}            # (mode, representation) -> per-query DataFrame (MC task)
PQ_FRS = {}
SIM_CACHE = {}
rows = []
for mode in MODES:
    q, g = EVAL[mode]["query"], EVAL[mode]["gallery"]
    reps = {r: base_sims(r, q, g, rng_len) for r in BASELINES}
    for name, m in MODELS.items():
        if m["mode"] == mode:
            reps[name] = embed_model(name, q["sequence"].tolist()) @ embed_model(name, g["sequence"].tolist()).T
    for rep, S in reps.items():
        pq = per_query(S, q, g, "MC"); PQ[(mode, rep)] = pq
        PQ_FRS[(mode, rep)] = per_query(S, q, g, "FRS")
        ch = CHANCE[(mode, "MC")]
        lo, hi = boot_mean_ci(pq["AP"].values)
        rec = {"mode": mode, "representation": rep, "kind": "model" if rep in MODELS else "baseline", "n_queries": len(pq),
               "AP": pq["AP"].mean(), "AP_lo": lo, "AP_hi": hi, "chance_AP": ch["chance_AP"], "lift": pq["AP"].mean() / ch["chance_AP"],
               "R@5": pq["R@5"].mean(), "chance_R@5": ch["chance_R@5"], "R@10": pq["R@10"].mean(), "chance_R@10": ch["chance_R@10"]}
        if rep in MODELS:
            Eq = embed_model(rep, q["sequence"].tolist())
            rec["mean_cos_heldout"], rec["eff_rank_heldout"] = spread(Eq)
            rec["ckpt_epoch"] = MODELS[rep]["ckpt"]["checkpoint_epoch"]
        rows.append(rec)
    # random-score row: AP/Recall at chance by construction
    ch = CHANCE[(mode, "MC")]
    rows.append({"mode": mode, "representation": "RANDOM (chance)", "kind": "chance", "n_queries": len(PQ[(mode, "RAW 650M")]),
                 "AP": ch["chance_AP"], "chance_AP": ch["chance_AP"], "lift": 1.0, "R@5": ch["chance_R@5"], "chance_R@5": ch["chance_R@5"],
                 "R@10": ch["chance_R@10"], "chance_R@10": ch["chance_R@10"]})
RETR = pd.DataFrame(rows)
cols = ["mode", "representation", "kind", "n_queries", "AP", "AP_lo", "AP_hi", "lift", "R@5", "R@10", "chance_R@5", "mean_cos_heldout", "eff_rank_heldout", "ckpt_epoch"]
for mode in MODES:
    print(f"\n=== mode: {mode}  | MC-active queries n={int((EVAL[mode]['query'].labels.str.contains('MC')).sum())} | gallery MC-active {CHANCE[(mode, 'MC')]['relevant']}/{CHANCE[(mode, 'MC')]['N_gallery']} "
          f"| chance AP={CHANCE[(mode, 'MC')]['chance_AP']:.3f}, chance R@5={CHANCE[(mode, 'MC')]['chance_R@5']:.3f}")
    print(RETR[RETR["mode"] == mode].sort_values("AP", ascending=False)[cols].round(3).to_string(index=False))
RETR.to_csv(f"{RES}/step5_mc_retrieval.csv", index=False)

---
## 3. Paired comparisons (same queries, bootstrap CI of the difference, Wilcoxon, Holm)

Each row compares two representations **on the same MC-active queries**. The primary quantity is Δ AP with its own 95 % bootstrap CI; the verdict is `A better` / `B better` only if that CI excludes 0, otherwise **`no difference detected`** (which is *not* evidence of equivalence — that would need a margin fixed in advance). The Wilcoxon signed-rank p-values are Holm-corrected over the whole family printed here.

In [ ]:
def paired(a, b, mode, label):
    A, B = PQ[(mode, a)]["AP"], PQ[(mode, b)]["AP"]
    assert (A.index == B.index).all()
    d = (A - B).values; rng = np.random.RandomState(1)
    bs = d[rng.randint(0, len(d), (N_BOOT, len(d)))].mean(1)
    lo, hi = np.percentile(bs, [2.5, 97.5])
    try: p = float(wilcoxon(d, alternative="two-sided", zero_method="wilcox")[1]) if np.any(d != 0) else 1.0
    except ValueError: p = 1.0
    verdict = f"{a} better" if lo > 0 else f"{b} better" if hi < 0 else "no difference detected"
    return {"comparison": label, "mode": mode, "A": a, "B": b, "n": len(d), "AP_A": A.mean(), "AP_B": B.mean(), "delta_AP": d.mean(),
            "ci_lo": lo, "ci_hi": hi, "p_wilcoxon": p, "verdict": verdict}

def have(mode, *reps): return all((mode, r) in PQ for r in reps)
fam = []
P = PRIMARY_MODE
for name in ("Primary", "Primary-MaxSpread", "Primary-MechEqual", "MechEqual-MaxSpread", "k-NN-Pairing", "Primary-Balanced", "HP2-Fusion"):
    if have(P, name):
        fam += [paired(name, "RAW 650M", P, f"{name} vs untrained ESM-2 650M"), paired(name, "AAC-20", P, f"{name} vs amino-acid composition"),
                paired(name, "LENGTH", P, f"{name} vs length only")]
if have(P, "Primary-150M"): fam += [paired("Primary-150M", "RAW 150M", P, "Primary-150M vs untrained ESM-2 150M")]
fam += [paired("RAW 650M", "AAC-20", P, "untrained ESM-2 650M vs amino-acid composition"), paired("RAW 650M", "LENGTH", P, "untrained ESM-2 650M vs length only"),
        paired("AAC-20", "LENGTH", P, "amino-acid composition vs length only")]
for a, b, lab in [("Primary", "HP1-Random", "HP1: real pairs vs random pairs (Primary vs HP1-Random)"), ("Primary", "Primary-MaxSpread", "selection rule: lowest loss vs most spread (Primary vs MaxSpread)"),
                  ("Primary", "Primary-Balanced", "MC resampling: Primary vs Balanced"), ("Primary", "Primary-MechEqual", "MC equal steps: Primary vs MechEqual"),
                  ("Primary-150M", "Primary", "backbone: 150M vs 650M (percentile pairs)"), ("k-NN-Pairing", "Primary", "pairing: k-NN vs percentile (650M)"),
                  ("k-NN-150M", "k-NN-Pairing", "backbone: 150M vs 650M (k-NN pairs)"), ("HP2-Fusion", "Primary", "HP2: fusion vs ESM-2 only")]:
    if have(P, a, b): fam.append(paired(a, b, P, lab))
for mode, model in (("extended", "Primary-Extended"), ("strict_ge4", "Primary-StrictGE4")):
    if have(mode, model): fam += [paired(model, "RAW 650M", mode, f"{model} vs untrained ESM-2 650M (mode {mode})"), paired(model, "AAC-20", mode, f"{model} vs amino-acid composition (mode {mode})")]
    fam.append(paired("RAW 650M", "AAC-20", mode, f"untrained ESM-2 650M vs composition (mode {mode})"))

PAIRED = pd.DataFrame(fam)
order = np.argsort(PAIRED["p_wilcoxon"].values); m_ = len(PAIRED); holm = np.empty(m_); run = 0.0
for rank, idx in enumerate(order):
    run = max(run, (m_ - rank) * PAIRED["p_wilcoxon"].values[idx]); holm[idx] = min(1.0, run)
PAIRED["p_holm"] = holm
print(f"family size = {m_} comparisons (Holm-corrected Wilcoxon)\n")
print(PAIRED[["comparison", "mode", "n", "AP_A", "AP_B", "delta_AP", "ci_lo", "ci_hi", "p_wilcoxon", "p_holm", "verdict"]].round(4).to_string(index=False))
PAIRED.to_csv(f"{RES}/step5_paired_comparisons.csv", index=False)

### 3b. Held-out collapse diagnostics and prototype geometry

Step 4 measured `mean_cos` / `eff_rank` on the **training** pool. Here they are measured on the **held-out fold-1** embeddings of each model (printed in the §2 tables as `mean_cos_heldout`, `eff_rank_heldout`) — 1.0 / 1 means every peptide sits at one point. The saved prototype distances are repeated for completeness; FRS–GEN ≈ 0 is descriptive (GEN is a mixed pool the loss never separates), not a collapse criterion.

In [ ]:
rows = []
for name, m in MODELS.items():
    ck = m["ckpt"]; mode = m["mode"]
    r = RETR[(RETR["mode"] == mode) & (RETR["representation"] == name)].iloc[0]
    rows.append({"model": name, "mode": mode, "epoch": ck["checkpoint_epoch"], "criterion_met": ck["criterion_met"],
                 "mean_cos_train": round(ck["mean_cos_at_ckpt"], 3), "mean_cos_heldout": round(r["mean_cos_heldout"], 3),
                 "eff_rank_train": round(ck["eff_rank_at_ckpt"], 1), "eff_rank_heldout": round(r["eff_rank_heldout"], 1),
                 "FRS-MC": round(ck["saved_distances"]["FRS-MC"], 3), "FRS-GEN": round(ck["saved_distances"]["FRS-GEN"], 4), "GEN-MC": round(ck["saved_distances"]["GEN-MC"], 3)})
GEOM = pd.DataFrame(rows); print(GEOM.to_string(index=False)); GEOM.to_csv(f"{RES}/step5_geometry_heldout.csv", index=False)

---
## 4. Supervised probe — MC-active vs FRS-only (build v4: repeated grouped CV, fold-mean AUROC, confound baselines)

Target: 1 = label set contains MC (MC-only + dual), 0 = FRS-only, over the mechanism-active peptides of the experiment mode (**both** folds, because the probe is cross-validated, not split by fold).

**What changed in v4 (STEP 08 findings N-19, N-21, N-22, N-23):**
* **Repeated grouped CV.** Whole `homology_group`s are assigned to 5 folds, stratified by label, **10 times with different fold assignments** (`group_folds`, own code, so the folds do not depend on the scikit-learn release). A single `GroupKFold(5)` split moved the ESM-2 650M AUROC between 0.59 and 0.66 depending on the scikit-learn version and the group-id order (N-22).
* **Headline metric = fold-mean AUROC** (mean of the 50 test-fold AUROCs). The pooled out-of-fold AUROC mixes five separately fitted intercepts and is biased (a single binary feature with exact AUROC 0.639 got 0.55-0.67 pooled; N-22). `AUROC_pooled` (AUROC of the repeat-averaged score) is kept as a secondary column.
* **CI = group bootstrap** over `homology_group`s (2000 resamples; every resample re-weights the test peptides of all 50 folds). It holds the fitted models fixed, so it understates the true uncertainty (refit variance is not included); read it together with `AUROC_rep_min/max`.
* **Confound baselines.** `SOURCE` (1 if `source_group == external_literature`: all FRS-only peptides are AnOxPePred, N-19) and `AAC-20 + LENGTH + SOURCE`, plus ESM-2 + those features. LENGTH matters through the **tripeptide share** (FRS-only 43 %, MC-active 16 %), not through the mean length (N-21).
* **Chance reference = NULL band**: fold-mean AUROC of 64-d pure-noise features over 20 noise seeds. The single noise matrix of Step 5 v3 (`RandomState(0)`) is kept as `RANDOM noise (seed 0)` but sits about 3 sd below its own null mean (N-23); do not read it as chance.

Features are standardised inside each training fold; logistic regression, C = 1, balanced class weights. A **leaky** run (repeated stratified random folds, which put near-duplicates on both sides) is added for ESM-2 650M and composition to *measure* how much leakage inflates the result.

In [ ]:
from sklearn.exceptions import ConvergenceWarning
import warnings; warnings.filterwarnings("ignore", category=ConvergenceWarning)

N_REP, N_FOLD, N_NULL = 10, 5, 20

def group_folds(y, g, k, seed):
    # whole homology groups -> k folds; groups that contain positives balance the positives, the rest balance the size.
    # Deterministic given `seed` and independent of the scikit-learn release.
    rng = np.random.RandomState(seed); ug = np.unique(g); rng.shuffle(ug)
    pos = {u: int(y[g == u].sum()) for u in ug}; tot = {u: int((g == u).sum()) for u in ug}
    ug = sorted(ug, key=lambda u: -(pos[u] * 10_000 + tot[u]))
    fp, fn = np.zeros(k), np.zeros(k); where = {}
    for u in ug:
        j = int(np.lexsort((fn, fp))[0]) if pos[u] > 0 else int(np.argmin(fn))
        where[u] = j; fp[j] += pos[u]; fn[j] += tot[u]
    f = np.array([where[x] for x in g])
    return [(np.where(f != j)[0], np.where(f == j)[0]) for j in range(k)]

def grouped_splits(y, g): return [group_folds(y, g, N_FOLD, 1000 + r) for r in range(N_REP)]
def leaky_splits(y):      return [list(StratifiedKFold(N_FOLD, shuffle=True, random_state=SEED + r).split(np.zeros(len(y)), y)) for r in range(N_REP)]

def fit_predict(Xtr, ytr, Xte):
    sc = StandardScaler().fit(Xtr)
    clf = LogisticRegression(max_iter=3000, C=1.0, class_weight="balanced").fit(sc.transform(Xtr), ytr)
    return clf.predict_proba(sc.transform(Xte))[:, 1]

def run_cv(X, y, splits):
    oof = np.full((len(splits), len(y)), np.nan); fa = np.full((len(splits), N_FOLD), np.nan)
    for r, folds in enumerate(splits):
        for k, (tr, te) in enumerate(folds):
            oof[r, te] = fit_predict(X[tr], y[tr], X[te])
            if len(set(y[te])) > 1: fa[r, k] = roc_auc_score(y[te], oof[r, te])
    assert np.isfinite(oof).all()
    return oof, fa

def boot_weights(g, n=N_BOOT, seed=0):
    # group bootstrap as multiplicity weights (n, n_samples): every peptide inherits the draw count of its homology group
    u, inv = np.unique(g, return_inverse=True)
    return np.random.RandomState(seed).multinomial(len(u), np.full(len(u), 1 / len(u)), size=n)[:, inv].astype(float)

def wauc(s, y, W):
    # weighted AUROC (ties = 1/2) for every row of W; NaN when a resample has no positive or no negative weight
    u, inv = np.unique(s, return_inverse=True); O = np.zeros((len(s), len(u))); O[np.arange(len(s)), inv] = 1
    P, N = (W * y) @ O, (W * (1 - y)) @ O
    num = (P * (np.cumsum(N, 1) - N + 0.5 * N)).sum(1); den = P.sum(1) * N.sum(1)
    return np.where(den > 0, num / np.where(den > 0, den, 1), np.nan)

_t = np.random.RandomState(1); _s, _y = _t.rand(50).round(1), _t.randint(0, 2, 50)
assert abs(wauc(_s, _y, np.ones((1, 50)))[0] - roc_auc_score(_y, _s)) < 1e-12, "weighted AUROC differs from scikit-learn"

def boot_foldmean(oof, y, splits, W):
    acc = [wauc(oof[r, te], y[te], W[:, te]) for r, folds in enumerate(splits) for _, te in folds]
    return np.nanmean(np.stack(acc, 1), 1)

def null_band(y, g, splits, dim=64):
    vals = np.array([np.nanmean(run_cv(np.random.RandomState(1000 + s).normal(size=(len(y), dim)), y, splits)[1]) for s in range(N_NULL)])
    return vals.mean(), np.percentile(vals, 2.5), np.percentile(vals, 97.5), vals.std(ddof=1)

PROBE, OOF, SPL, WB = [], {}, {}, {}
for mode in MODES:
    d = POOL[mode]; seqs = d["sequence"].tolist(); y = d["labels"].str.contains("MC").astype(int).values; g = d["homology_group"].values.astype(int)
    src = (d["source_group"] == "external_literature").astype(float).values.reshape(-1, 1)
    Xs = {f"ESM-2 {sc}": np.stack([ESM[sc][s] for s in seqs]) for sc in SCALES}
    Xs["AAC-20"] = np.stack([AAC[s] for s in seqs]); Xs["DESC-14"] = np.stack([DESC[s] for s in seqs])
    Xs["LENGTH"] = np.array([[LENGTH[s], math.log(LENGTH[s])] for s in seqs]); Xs["AAC-20 + LENGTH"] = np.hstack([Xs["AAC-20"], Xs["LENGTH"]])
    Xs["SOURCE"] = src; Xs["AAC-20 + LENGTH + SOURCE"] = np.hstack([Xs["AAC-20 + LENGTH"], src])
    for sc in ("650M", "150M"): Xs[f"ESM-2 {sc} + AAC-20 + LENGTH + SOURCE"] = np.hstack([Xs[f"ESM-2 {sc}"], Xs["AAC-20 + LENGTH + SOURCE"]])
    Xs["RANDOM noise (seed 0)"] = np.random.RandomState(0).normal(size=(len(y), 64))
    L = d["length"].values if "length" in d.columns else np.array([LENGTH[s] for s in seqs])
    print(f"\n=== mode {mode}: n={len(y)}  MC-active={int(y.sum())}  FRS-only={int((1 - y).sum())}  homology groups={len(np.unique(g))}")
    print(pd.crosstab(d["labels"], d["source_group"]).to_string())
    print(f"  tripeptide share: FRS-only {np.mean(L[y == 0] == 3):.1%}  MC-active {np.mean(L[y == 1] == 3):.1%}   mean length: FRS-only {L[y == 0].mean():.2f}  MC-active {L[y == 1].mean():.2f}   "
          f"exact SOURCE-only AUROC (one binary feature, no fitting) = {roc_auc_score(y, src[:, 0]):.3f}")
    SP, W = grouped_splits(y, g), boot_weights(g); SPL[mode], WB[mode] = SP, W
    for name, X in Xs.items():
        o, fa = run_cv(X, y, SP); OOF[(mode, name)] = (y, o, fa)
        b = boot_foldmean(o, y, SP, W); lo, hi = np.percentile(b, [2.5, 97.5]); rep = np.nanmean(fa, 1)
        PROBE.append({"mode": mode, "features": name, "dim": X.shape[1], "n": len(y), "n_MC": int(y.sum()), "AUROC": float(np.nanmean(fa)), "lo": lo, "hi": hi,
                      "AUROC_rep_sd": rep.std(ddof=1), "AUROC_rep_min": rep.min(), "AUROC_rep_max": rep.max(), "AUROC_pooled": roc_auc_score(y, o.mean(0)),
                      "cv": f"repeated grouped CV ({N_FOLD} folds x {N_REP}, homology_group)"})
    m, lo, hi, sd = null_band(y, g, SP)
    PROBE.append({"mode": mode, "features": f"NULL band: RANDOM noise 64-d, {N_NULL} seeds", "dim": 64, "n": len(y), "n_MC": int(y.sum()), "AUROC": m, "lo": lo, "hi": hi,
                  "AUROC_rep_sd": sd, "AUROC_rep_min": np.nan, "AUROC_rep_max": np.nan, "AUROC_pooled": np.nan, "cv": f"repeated grouped CV ({N_FOLD} folds x {N_REP}, homology_group)"})
    LS = leaky_splits(y)
    for name in ("ESM-2 650M", "AAC-20"):
        o, fa = run_cv(Xs[name], y, LS); b = boot_foldmean(o, y, LS, W); lo, hi = np.percentile(b, [2.5, 97.5]); rep = np.nanmean(fa, 1)
        PROBE.append({"mode": mode, "features": name, "dim": Xs[name].shape[1], "n": len(y), "n_MC": int(y.sum()), "AUROC": float(np.nanmean(fa)), "lo": lo, "hi": hi,
                      "AUROC_rep_sd": rep.std(ddof=1), "AUROC_rep_min": rep.min(), "AUROC_rep_max": rep.max(), "AUROC_pooled": roc_auc_score(y, o.mean(0)),
                      "cv": f"LEAKY stratified random folds ({N_FOLD} folds x {N_REP})"})
PR = pd.DataFrame(PROBE)
print("\n" + PR.round(4).to_string(index=False)); PR.to_csv(f"{RES}/step5_probe.csv", index=False)
print("\nAUROC = fold-mean AUROC over the 50 test folds (headline); AUROC_pooled = AUROC of the repeat-averaged out-of-fold score (secondary, biased by fold intercepts).")
print("CI = group bootstrap over homology groups with the fitted models held fixed -> narrower than the true uncertainty; compare with AUROC_rep_min/max and the NULL band.")
print("Chance reference = the NULL band row (20 noise seeds); 'RANDOM noise (seed 0)' is one low draw, not chance (N-23).")

### 4b. Paired differences between feature sets, scale ordering, and the leakage measurement
Differences are computed on the **same folds and the same bootstrap resamples** (fold-mean AUROC), so they are paired. `repeats_delta_gt_0` = in how many of the 10 fold assignments the difference has the sign of the point estimate's direction (model minus reference > 0).

In [ ]:
rows = []
def paired(mode, a, b):
    y, oa, fa_a = OOF[(mode, a)]; _, ob, fa_b = OOF[(mode, b)]
    dlt = boot_foldmean(oa, y, SPL[mode], WB[mode]) - boot_foldmean(ob, y, SPL[mode], WB[mode]); lo, hi = np.percentile(dlt, [2.5, 97.5])
    rep = np.nanmean(fa_a, 1) - np.nanmean(fa_b, 1)
    rows.append({"mode": mode, "A": a, "B": b, "delta_AUROC": float(np.nanmean(fa_a) - np.nanmean(fa_b)), "ci_lo": lo, "ci_hi": hi,
                 "repeats_delta_gt_0": int((rep > 0).sum()), "verdict": f"{a} better" if lo > 0 else f"{b} better" if hi < 0 else "no difference detected"})
for mode in MODES:
    for name in [f"ESM-2 {sc}" for sc in SCALES] + ["DESC-14", "LENGTH", "AAC-20 + LENGTH", "SOURCE", "AAC-20 + LENGTH + SOURCE"]:
        paired(mode, name, "AAC-20")
    for a, b in [("ESM-2 3B", "ESM-2 35M"), ("ESM-2 650M", "ESM-2 35M"), ("ESM-2 650M", "ESM-2 150M"), ("ESM-2 3B", "ESM-2 650M")]:
        paired(mode, a, b)
    # the decisive controls: does ESM-2 add anything on top of composition + length + source?
    paired(mode, "AAC-20 + LENGTH + SOURCE", "AAC-20 + LENGTH")
    paired(mode, "ESM-2 650M", "SOURCE")
    for sc in ("650M", "150M"):
        paired(mode, f"ESM-2 {sc} + AAC-20 + LENGTH + SOURCE", "AAC-20 + LENGTH + SOURCE")
        paired(mode, f"ESM-2 {sc}", "AAC-20 + LENGTH + SOURCE")
PD = pd.DataFrame(rows); print(PD.round(4).to_string(index=False)); PD.to_csv(f"{RES}/step5_probe_paired.csv", index=False)

print("\nLeakage measurement (fold-mean AUROC, grouped vs leaky random folds, 10 repeats each):")
for mode in MODES:
    for name in ("ESM-2 650M", "AAC-20"):
        a = PR[(PR["mode"] == mode) & (PR["features"] == name) & PR["cv"].str.startswith("repeated grouped")]["AUROC"].iloc[0]
        b = PR[(PR["mode"] == mode) & (PR["features"] == name) & PR["cv"].str.startswith("LEAKY")]["AUROC"].iloc[0]
        print(f"  {mode:<11} {name:<11} grouped={a:.3f}  leaky={b:.3f}  inflation={b - a:+.3f}")

fig, axes = plt.subplots(1, len(MODES), figsize=(6 * len(MODES), 4.6), sharey=True)
for ax, mode in zip(np.atleast_1d(axes), MODES):
    sub = PR[(PR["mode"] == mode) & PR["cv"].str.startswith("repeated grouped") & ~PR["features"].str.startswith("NULL")].reset_index(drop=True)
    nb = PR[(PR["mode"] == mode) & PR["features"].str.startswith("NULL")].iloc[0]
    ax.axhspan(nb["lo"], nb["hi"], color="lightgray", alpha=0.6, label="noise null band (20 seeds, 2.5-97.5 %)")
    ax.errorbar(range(len(sub)), sub["AUROC"], yerr=[np.clip(sub["AUROC"] - sub["lo"], 0, None), np.clip(sub["hi"] - sub["AUROC"], 0, None)], fmt="o", capsize=3)
    ax.axhline(0.5, ls="--", c="gray"); ax.set_xticks(range(len(sub))); ax.set_xticklabels(sub["features"], rotation=70, ha="right", fontsize=7)
    ax.set_title(f"fold-mean AUROC (repeated grouped CV), mode {mode}", fontsize=9); ax.grid(alpha=0.3)
np.atleast_1d(axes)[0].set_ylabel("AUROC (95% group-bootstrap CI)"); np.atleast_1d(axes)[0].legend(fontsize=7, loc="upper left")
plt.tight_layout(); plt.savefig(f"{RES}/step5_probe.png", dpi=120); plt.show()

---
## 5. Conclusions — generated from the tables above

Nothing below is hand-written; every sentence is a function of `RETR`, `PAIRED`, `GEOM`, `PR` and `PD`. A statement is printed only if the table supports it.

In [ ]:
P = PRIMARY_MODE
print("=" * 80); print("STEP 5 CONCLUSIONS (generated)"); print("=" * 80)
q_n = int(EVAL[P]["query"]["labels"].str.contains("MC").sum()); ch = CHANCE[(P, "MC")]
print(f"\n1. EVALUATION POWER. Primary mode '{P}': MC retrieval uses n={q_n} MC-active queries against a gallery with {ch['relevant']}/{ch['N_gallery']} relevant items "
      f"(chance AP={ch['chance_AP']:.3f}, chance R@5={ch['chance_R@5']:.3f}). FRS retrieval is saturated (chance R@5={CHANCE[(P, 'FRS')]['chance_R@5']:.3f}) and carries no information.")
n_only = int((EVAL[P]["query"]["labels"] == "MC").sum()); n_only_g = int((EVAL[P]["gallery"]["labels"] == "MC").sum())
print(f"   MC-only peptides: {n_only} in the query fold, {n_only_g} in the gallery fold -> the MC prototype is learned from dual peptides only.")

tr = RETR[(RETR["mode"] == P) & (RETR["kind"] == "model")]; bl = RETR[(RETR["mode"] == P) & (RETR["kind"] == "baseline")]
best_b = bl.sort_values("AP", ascending=False).iloc[0]
print(f"\n2. BASELINES. Best untrained baseline: {best_b['representation']} (AP={best_b['AP']:.3f} [{best_b['AP_lo']:.3f},{best_b['AP_hi']:.3f}], lift {best_b['lift']:.2f}x chance).")
beats = PAIRED[(PAIRED["B"].isin(["RAW 650M", "RAW 150M"])) & (PAIRED["A"].isin(MODELS))]
n_b = int((beats["verdict"].str.endswith("better") & (beats["delta_AP"] > 0)).sum()); n_w = int(((beats["delta_AP"] < 0) & (beats["verdict"] != "no difference detected")).sum())
print(f"   Trained models vs their untrained ESM-2 backbone: {n_b} better, {n_w} worse, {len(beats) - n_b - n_w} no difference detected (paired 95% CI).")
for _, r in PAIRED[PAIRED["A"].isin(MODELS) & PAIRED["B"].isin(["RAW 650M", "RAW 150M"])].iterrows():
    print(f"     {r['A']:<22} vs {r['B']:<9} (mode {r['mode']}): dAP={r['delta_AP']:+.3f} [{r['ci_lo']:+.3f},{r['ci_hi']:+.3f}]  -> {r['verdict']}")
cmp_ = PAIRED[PAIRED["comparison"] == "untrained ESM-2 650M vs amino-acid composition"].iloc[0]
print(f"   untrained ESM-2 650M vs composition: dAP={cmp_['delta_AP']:+.3f} [{cmp_['ci_lo']:+.3f},{cmp_['ci_hi']:+.3f}] -> {cmp_['verdict']}")

print(f"\n3. HELD-OUT COLLAPSE. mean_cos on fold-1 embeddings: min {GEOM['mean_cos_heldout'].min():.3f}, max {GEOM['mean_cos_heldout'].max():.3f}; "
      f"{int((GEOM['mean_cos_heldout'] > 0.95).sum())}/{len(GEOM)} checkpoints have mean_cos > 0.95 (near single-point embeddings).")
hp = PAIRED[PAIRED["comparison"].str.startswith("HP1")]
if len(hp):
    r = hp.iloc[0]; print(f"\n4. HP1 (real vs random pairs): dAP={r['delta_AP']:+.3f} [{r['ci_lo']:+.3f},{r['ci_hi']:+.3f}] -> {r['verdict']}. This is a single seed on {r['n']} queries.")
sel = PAIRED[PAIRED["comparison"].str.startswith("selection rule")]
if len(sel):
    r = sel.iloc[0]; print(f"   Selection rule (Primary vs MaxSpread): dAP={r['delta_AP']:+.3f} [{r['ci_lo']:+.3f},{r['ci_hi']:+.3f}] -> {r['verdict']}.")

print(f"\n5. SUPERVISED PROBE (MC-active vs FRS-only; repeated grouped CV by homology_group, fold-mean AUROC), mode {P}:")
GR = PR[PR["cv"].str.startswith("repeated grouped")]
sub = GR[GR["mode"] == P].set_index("features")
nbp = sub[sub.index.str.startswith("NULL")].iloc[0]
print(f"   chance reference (NULL band, noise, 20 seeds): {nbp['AUROC']:.3f} [{nbp['lo']:.3f},{nbp['hi']:.3f}]")
for f in ("SOURCE", "LENGTH", "AAC-20", "AAC-20 + LENGTH", "AAC-20 + LENGTH + SOURCE", "DESC-14", "ESM-2 35M", "ESM-2 150M", "ESM-2 650M", "ESM-2 3B", "RANDOM noise (seed 0)"):
    r = sub.loc[f]; flag = "above" if r["AUROC"] > nbp["hi"] else "inside/below"
    print(f"   {f:<26} AUROC={r['AUROC']:.3f} [{r['lo']:.3f},{r['hi']:.3f}]  repeat range [{r['AUROC_rep_min']:.3f},{r['AUROC_rep_max']:.3f}]  vs null band: {flag}")
def pd_row(a, b): return PD[(PD["mode"] == P) & (PD["A"] == a) & (PD["B"] == b)].iloc[0]
for a, b in [("ESM-2 650M", "AAC-20"), ("ESM-2 650M", "SOURCE"), ("ESM-2 650M", "AAC-20 + LENGTH + SOURCE"),
             ("ESM-2 650M + AAC-20 + LENGTH + SOURCE", "AAC-20 + LENGTH + SOURCE"), ("ESM-2 150M + AAC-20 + LENGTH + SOURCE", "AAC-20 + LENGTH + SOURCE"),
             ("AAC-20 + LENGTH + SOURCE", "AAC-20 + LENGTH")]:
    r = pd_row(a, b); print(f"   {a} minus {b}: dAUROC={r['delta_AUROC']:+.3f} [{r['ci_lo']:+.3f},{r['ci_hi']:+.3f}], positive in {r['repeats_delta_gt_0']}/{N_REP} fold assignments -> {r['verdict']}")
print("   'no difference detected' is NOT 'equivalent'; group-bootstrap CIs hold the models fixed and are narrower than the true uncertainty.")
if sub.loc["SOURCE", "AUROC"] > sub.loc["ESM-2 650M", "AUROC"]:
    print("   CONFOUND: the single binary SOURCE feature is at least as good as ESM-2 650M (all FRS-only peptides come from AnOxPePred); length acts through the tripeptide share, not the mean length.")
leak = PR[(PR["mode"] == P)]
for name in ("ESM-2 650M", "AAC-20"):
    a = leak[(leak["features"] == name) & leak["cv"].str.startswith("repeated grouped")]["AUROC"].iloc[0]; b = leak[(leak["features"] == name) & leak["cv"].str.startswith("LEAKY")]["AUROC"].iloc[0]
    print(f"   Leakage ({name}): grouped={a:.3f} vs leaky random folds={b:.3f} -> inflation {b - a:+.3f}.")
print("\n6. WHAT THIS DOES NOT SHOW: one seed, one fold direction, tens of MC queries; no external validation set; "
      "assay-level labels are noisy; the GEN class is not evaluated for activity.")
print("=" * 80)

---
## Outputs

| File | Contents |
|---|---|
| `step5_mc_retrieval.csv` | every representation × mode: AP (CI), lift over chance, Recall@5/10, held-out `mean_cos` / `eff_rank` |
| `step5_paired_comparisons.csv` | paired ΔAP with CI, Wilcoxon p, Holm-corrected p, verdict |
| `step5_geometry_heldout.csv` | training-pool vs held-out spread, saved prototype distances |
| `step5_probe.csv`, `step5_probe_paired.csv`, `step5_probe.png` | repeated-grouped-CV probe (fold-mean AUROC, group-bootstrap CI, NULL band), paired differences incl. the SOURCE / LENGTH confound controls, leakage measurement |

**What the probe numbers are (v4):** fold-mean AUROC over 50 test folds (5 folds × 10 fold assignments); single-split values from earlier builds (for example strict ESM-2 650M 0.593) are one draw of that distribution (0.57-0.66) and must not be quoted on their own (N-22).

**Next:** seed spread (retrain Primary and HP1-Random with several seeds and report the spread of ΔAP) and the reverse fold direction are in Step 6; confound controls beyond this probe (AnOxPePred-only subset, length-matched subsampling, learning curve) are in `STEP 08`.